# 08c — Diagnose incorrect acceptance after 08b

Reuse saved regression/classification test predictions and the **original calibration-selected cutoffs**.
No model is trained, loaded or scored here; no cutoff is selected or changed. Empirical is the primary
analysis and grid_empirical is a separate supplement. A good answer has true relative error **<=25%**.

This is exploratory test diagnosis, not a new independent evaluation. Query IDs recur across merge
rates and seeds. All counts below are descriptive **query × rate × seed cases**, not independent trials.
No confidence intervals or significance tests are calculated from these repeated cases. Answers with
truth below10 were excluded by 08b; zero-count-query performance cannot be inferred.

The notebook can use a complete 08b run directory or its prediction ZIP. The ZIP alone supports
decisions, paired errors and severity. Query-size and source-contrast analyses require additional,
matching metadata/features. Missing inputs are reported explicitly instead of guessed.


## Settings

Set **run_dir** to the completed 08b directory, or **predictions_zip** to its saved ZIP. Leave both
blank only when output_root contains exactly one completed 08b run. Multiple runs require an explicit
choice; nothing is selected from test performance. Outputs go to a new 08c directory, never the source.

Optional `metadata_file` is the original queries-meta.csv; its `scale` is reused verbatim as query size.
Optional `feature_table` is a CSV with qId, mergeRate and the available original features. A full run
can rebuild the existing features from fingerprint-checked saved inputs; this does not run the model.
Source-contrast quantile bins require validated per-seed training IDs and training contrast values.
`training_table` can provide qId, src_edge_contrast and optional seed/mergeRate columns when the feature
CSV contains only test cases. No bin boundaries are fitted on test values.

`save_decisions=false` keeps compact summaries. Set it to true to also save compressed per-query
decisions and paired cases, including the four outcome classes. These optional files can be large.

Low-support flags (100 cases or 30 accepted cases by default) are descriptive reminders, not proof that
larger groups are stable. Changing these flags does not change any decision or gate.


In [ ]:
import hashlib
import importlib
import io
import json
import os
from pathlib import Path
import sys
import time
from uuid import uuid4
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODELS = ('regression', 'classification')
SEEDS = (0, 1, 2)
POLICIES = ('empirical', 'grid_empirical')
EPSILON = 0.25
FEATURES_18 = ['area_cells','n_cells','n_partial','side_frac','log_est','log_full','bound_rel',
               'edge_share','edge_contrast','in_out','empty_partial','est_frac','density_ratio',
               'merge_rate','n_src_overlap','src_res_min','src_res_max','src_edge_contrast']
OUTCOMES = ['good_accepted','bad_accepted','good_rejected','bad_rejected']

def _inferred_repo():
    try:
        name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        root = name.split('/ml/notebooks/')[0]
        return root if root.startswith('/Workspace/') else '/Workspace' + root
    except Exception:
        return os.environ.get('REPO_ROOT', os.getcwd())

def _settings():
    repo = _inferred_repo()
    parts = Path(repo).parts
    output = ''
    if 'Users' in parts and len(parts)>parts.index('Users')+1:
        index = parts.index('Users')+1
        output = str(Path(*parts[:index+1])/'synopsislake-results')
    return dict(run_dir='',predictions_zip='',output_root=output,repo_root=repo,feature_table='',
                metadata_file='',training_table='',min_group_cases='100',min_group_accepted='30',
                examples_per_category='3',save_decisions='false')

DEFAULTS = _settings()
try:
    for key,value in DEFAULTS.items():
        try: dbutils.widgets.get(key)
        except Exception: dbutils.widgets.text(key,value)
    get = dbutils.widgets.get
except NameError:
    def get(key): return os.environ.get(key.upper(),DEFAULTS[key])

def _csv(source): return pd.read_csv(source,float_precision='round_trip')
def _sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()
def _write_json(path,data): Path(path).write_text(json.dumps(data,ensure_ascii=False,indent=2,default=str))
def _ratio(n,d): return float(n)/float(d) if d else np.nan
def _bool(value):
    if isinstance(value,(bool,np.bool_)): return bool(value)
    if str(value).lower() in ('true','1'): return True
    if str(value).lower() in ('false','0'): return False
    raise ValueError(f'Expected a boolean, got {value!r}')


## Load saved decisions and audit the source

CSV floats use round-trip parsing. Acceptance is **score <= saved cutoff**, including all ties.
There is no tolerance added to this comparison. A saved accept_none gate rejects every case.
Any count, label, cohort or gate mismatch stops the analysis before publishing successful results.


In [ ]:
class SavedSource:
    def __init__(self,config):
        self.directory=None
        self.zip=None
        self.artifact_hashes={}
        run=str(config.get('run_dir','')).strip()
        bundle=str(config.get('predictions_zip','')).strip()
        if run and bundle: raise ValueError('Choose run_dir OR predictions_zip, not both')
        if not run and not bundle:
            candidates=[]
            for candidate in sorted(Path(config['output_root']).expanduser().glob('*-objective-compare-*')):
                try: state=json.loads((candidate/'run.json').read_text())
                except (OSError,ValueError): continue
                if state.get('status')=='complete' and state.get('study')=='objective-compare': candidates.append(candidate)
            if len(candidates)!=1:
                choices='\n'.join(str(p) for p in candidates) or '(none found)'
                raise ValueError('Set run_dir or predictions_zip explicitly. Compatible completed directories:\n'+choices)
            run=str(candidates[0])
        if bundle:
            self.path=Path(bundle).expanduser().resolve()
            self.zip=ZipFile(self.path)
            names=self.zip.namelist()
            if len(names)!=len(set(names)): raise ValueError('ZIP contains duplicate entry names')
            if any(Path(name).is_absolute() or '..' in Path(name).parts for name in names):
                raise ValueError('ZIP contains unsafe paths')
        else:
            self.path=Path(run).expanduser().resolve()
            self.directory=self.path
            if not self.path.is_dir(): raise FileNotFoundError(f'No run directory at {self.path}')
    def has(self,name):
        return name in self.zip.namelist() if self.zip else (self.directory/name).is_file()
    def read(self,name):
        raw=self.zip.read(name) if self.zip else (self.directory/name).read_bytes()
        self.artifact_hashes[name]=hashlib.sha256(raw).hexdigest()
        return raw
    def frame(self,name): return _csv(io.BytesIO(self.read(name)))
    def state(self,name): return json.loads(self.read(name))
    def close(self):
        if self.zip: self.zip.close()

def validate_source_state(state):
    if state.get('status')!='complete' or state.get('study')!='objective-compare':
        raise ValueError('Source must be a completed notebook08b objective-compare run')
    cfg=state['config']
    seeds=[int(v) for v in str(cfg['seeds']).split(',')]
    if sorted(seeds)!=list(SEEDS) or len(seeds)!=3: raise ValueError('Use the original seeds0,1,2')
    if [float(v) for v in str(cfg['epsilons']).split(',')]!=[EPSILON]: raise ValueError('Expected original25% boundary')
    if float(cfg['delta'])!=.05: raise ValueError('Expected original5% risk target')
    budgets=[int(v) for v in str(cfg['tree_budgets']).split(',')]
    if len(budgets)!=1: raise ValueError('Expected one original shared tree budget')
    if set(state.get('model_kinds',[]))!=set(MODELS): raise ValueError('Both original model kinds are required')
    for model in MODELS:
        if state.get('feature_definitions',{}).get(model)!=FEATURES_18:
            raise ValueError(f'{model}: source is not the original ordered18feature no-SA model')
    return budgets[0]

def load_predictions(source,state,budget):
    frames=[]
    for model in MODELS:
        for seed in SEEDS:
            name=f'models/{model}/trees_{budget}/seed_{seed}/test_predictions.csv'
            frame=source.frame(name)
            required={'qId','mergeRate','relErr','score','bad_label'}
            if not required.issubset(frame): raise ValueError(f'Missing prediction columns in {name}')
            if frame.empty or frame[['qId','mergeRate']].duplicated().any(): raise ValueError(f'Empty/duplicate predictions in {name}')
            if not np.isfinite(frame[['qId','mergeRate','relErr','score']].to_numpy(float)).all(): raise ValueError('Nonfinite predictions')
            if (frame.relErr<0).any() or not np.equal(frame.qId,np.floor(frame.qId)).all(): raise ValueError('Invalid qId/error')
            if not np.array_equal(frame.bad_label.to_numpy(),(frame.relErr>EPSILON).astype(int).to_numpy()): raise ValueError('Saved bad labels differ from strict25% boundary')
            if model=='classification' and not frame.score.between(0,1).all(): raise ValueError('Classifier scores are not probabilities')
            frame=frame[['qId','mergeRate','relErr','score','bad_label']].copy()
            frame['model_kind']=model;frame['seed']=seed
            frames.append(frame)
    predictions=pd.concat(frames,ignore_index=True)
    if predictions.groupby(['qId','mergeRate']).relErr.nunique().gt(1).any(): raise ValueError('True errors differ across repeated seed/model cases')
    expected_rates=set(state['calibration_family']['rates'])
    for (model,seed),part in predictions.groupby(['model_kind','seed']):
        if set(part.mergeRate)!=expected_rates: raise ValueError('Prediction rates differ from the saved calibration family')
        reference=None
        for rate,rate_part in part.groupby('mergeRate'):
            ids=set(rate_part.qId)
            if reference is None: reference=ids
            elif ids!=reference: raise ValueError('Eligible query cohort differs across merge rates')
    for seed in SEEDS:
        left=predictions[(predictions.model_kind=='regression')&(predictions.seed==seed)].sort_values(['qId','mergeRate'])
        right=predictions[(predictions.model_kind=='classification')&(predictions.seed==seed)].sort_values(['qId','mergeRate'])
        columns=['qId','mergeRate','relErr','bad_label']
        pd.testing.assert_frame_equal(left[columns].reset_index(drop=True),right[columns].reset_index(drop=True),check_dtype=False)
    return predictions

def load_gates(source,state,budget):
    gates=source.frame('gate_details.csv')
    required={'model_kind','tree_budget','seed','score','policy','eps','scope','mergeRate','cutoff','accept_none',
              'test_count','accepted_count','violation_count'}
    if not required.issubset(gates): raise ValueError('gate_details.csv is missing required columns')
    gates=gates[gates.score.eq('learned')&gates.policy.isin(POLICIES)].copy()
    if not gates.tree_budget.eq(budget).all() or not gates.eps.eq(EPSILON).all() or not gates.scope.eq('per_rate').all():
        raise ValueError('Saved gate settings differ')
    keys=['model_kind','seed','mergeRate','policy']
    if gates.duplicated(keys).any(): raise ValueError('Duplicate saved gate')
    expected={(model,seed,float(rate),policy) for model in MODELS for seed in SEEDS
              for rate in state['calibration_family']['rates'] for policy in POLICIES}
    if set(gates[keys].itertuples(index=False,name=None))!=expected: raise ValueError('Missing or unexpected saved gates')
    gates['accept_none']=gates.accept_none.map(_bool)
    if (gates.accept_none & gates.cutoff.notna()).any(): raise ValueError('accept_none must have an empty saved cutoff')
    if (~gates.accept_none & ~np.isfinite(gates.cutoff)).any(): raise ValueError('An accepting gate must have a finite cutoff')
    for name in ['test_count','accepted_count','violation_count']:
        if not np.isfinite(gates[name]).all() or (gates[name]<0).any() or not np.equal(gates[name],np.floor(gates[name])).all():
            raise ValueError(f'Invalid gate counts: {name}')
    return gates

def reconstruct_decisions(predictions,gates):
    keys=['model_kind','seed','mergeRate']
    decisions=predictions.merge(gates,on=keys,how='inner',validate='many_to_many',suffixes=('','_gate'))
    if len(decisions)!=len(predictions)*len(POLICIES): raise ValueError('A prediction is missing a policy gate')
    decisions['accepted']=~decisions.accept_none & decisions.score.le(decisions.cutoff)
    decisions['good']=decisions.relErr.le(EPSILON)
    decisions['outcome']=np.select([decisions.good&decisions.accepted,~decisions.good&decisions.accepted,
                                    decisions.good&~decisions.accepted],OUTCOMES[:3],default=OUTCOMES[3])
    audits=[]
    for identity,part in decisions.groupby(keys+['policy'],sort=True):
        row=dict(zip(keys+['policy'],identity))
        n,accepted,bad=len(part),int(part.accepted.sum()),int((part.accepted&~part.good).sum())
        for name,value in [('test_count',n),('accepted_count',accepted),('violation_count',bad)]:
            if not part[name].eq(value).all(): raise ValueError(f'Saved gate count mismatch: {row}, {name}, reconstructed={value}')
        row.update(test_count=n,accepted_count=accepted,violation_count=bad,matched=True)
        audits.append(row)
    keep=['model_kind','seed','mergeRate','policy','qId','relErr','score','cutoff','accept_none','accepted','good','outcome']
    return decisions[keep].copy(),pd.DataFrame(audits)

def pair_decisions(decisions):
    keys=['policy','seed','qId','mergeRate']
    columns=keys+['relErr','good','accepted','score','cutoff','accept_none','outcome']
    left=decisions[decisions.model_kind.eq('regression')][columns]
    right=decisions[decisions.model_kind.eq('classification')][columns]
    paired=left.merge(right,on=keys,validate='one_to_one',suffixes=('_regression','_classification'))
    if len(paired)*2!=len(decisions): raise ValueError('Models do not have the same paired cases')
    if not paired.relErr_regression.eq(paired.relErr_classification).all() or not paired.good_regression.eq(paired.good_classification).all():
        raise ValueError('Paired true errors differ')
    paired=paired.rename(columns={'relErr_regression':'relErr','good_regression':'good'}).drop(columns=['relErr_classification','good_classification'])
    r,c=paired.accepted_regression,paired.accepted_classification
    paired['acceptance_overlap']=np.select([r&c,r&~c,~r&c],['both','regression_only','classification_only'],default='neither')
    return paired


## Optional query metadata and original features

The existing size group is `queries-meta.csv.scale`: it ranks original query widths, not histogram
cell counts or newly fitted area bins. Width/height are retained as context. Source contrast is
query-specific and must agree across merge rates. Its quartile boundaries are fitted separately for
each seed using only that seed's original training query IDs, deduplicated across merge rates.
Exterior bounds are infinite, so out-of-training-range test values remain in fixed bins.
Pooled C1/C2/etc. labels are per-seed relative quantile groups whose absolute boundaries can differ;
see `training_bin_edges.csv` for each seed's fixed boundaries.

Missing source histograms, feature cache or verified training splits disable only the unsupported
groupings. Their absence does not stop the decision and severity analyses.


In [ ]:
def _canonical_splits(qids,seed):
    ids=np.random.default_rng(seed).permutation(np.unique(qids))
    n=len(ids)
    return {'train':set(ids[:n//2]),'calibration':set(ids[n//2:int(.7*n)]),'test':set(ids[int(.7*n):])}

def _validate_splits(frame,predictions,state):
    required={'seed','qId','split'}
    if not required.issubset(frame) or frame.duplicated(['seed','qId']).any(): raise ValueError('Invalid saved split manifest')
    if set(frame.seed)!=set(SEEDS) or not set(frame.split).issubset({'train','calibration','test'}): raise ValueError('Unexpected saved splits')
    universe=None
    for seed in SEEDS:
        part=frame[frame.seed.eq(seed)]
        ids=set(part.qId)
        if universe is None: universe=ids
        elif ids!=universe: raise ValueError('Split query universes differ across seeds')
        expected=_canonical_splits(part.qId,seed)
        for label,wanted in expected.items():
            if set(part.loc[part.split.eq(label),'qId'])!=wanted: raise ValueError('Saved split differs from the original50/20/30 query split')
        actual=set(predictions.loc[predictions.seed.eq(seed),'qId'])
        if actual!=expected['test']: raise ValueError('Saved test predictions disagree with split IDs')
    if state.get('unique_queries') is not None and len(universe)!=int(state['unique_queries']): raise ValueError('Saved split universe size differs')
    return frame[['seed','qId','split']].copy()

def _feature_identity(frame,state):
    if not {'qId','mergeRate'}.issubset(frame): raise ValueError('Feature table needs qId and mergeRate')
    if frame.duplicated(['qId','mergeRate']).any(): raise ValueError('Feature table has duplicate query/rate keys')
    if not np.isfinite(frame[['qId','mergeRate']].to_numpy(float)).all(): raise ValueError('Nonfinite feature keys')
    if 'merge_rate' in frame and not np.allclose(frame.merge_rate,frame.mergeRate,rtol=0,atol=1e-14): raise ValueError('merge_rate differs from mergeRate')
    present=[name for name in FEATURES_18 if name in frame]
    if present and not np.isfinite(frame[present].to_numpy(float)).all(): raise ValueError('Nonfinite supplied features')
    if 'src_edge_contrast' in frame and frame.groupby('qId').src_edge_contrast.nunique().gt(1).any():
        raise ValueError('Source contrast varies by merge rate for a query')
    return frame

def load_diagnostics(config,source,state,predictions):
    missing=[];provenance={};features=None;metadata=None;splits=None
    if source.has('splits.csv'):
        splits=_validate_splits(source.frame('splits.csv'),predictions,state)
        provenance['split_source']='saved splits.csv'
    feature_path=str(config.get('feature_table','')).strip()
    metadata_path=str(config.get('metadata_file','')).strip()
    candidates=[]
    if source.directory:
        candidates.append((source.directory/'inputs/data',source.directory/'inputs/run','saved input snapshot'))
    cfg=state['config']
    if cfg.get('data_root') and cfg.get('sample_root'):
        candidates.append((Path(cfg['data_root'])/cfg['dataset'],Path(cfg['sample_root'])/cfg['dataset'],'original matching inputs'))
    verified=None
    expected=state.get('input_sha256',{})
    for data,samples,label in candidates:
        paths={name:(data if name.startswith('data/') else samples)/Path(name).name for name in expected}
        if paths and all(path.is_file() for path in paths.values()):
            if any(_sha(path)!=expected[name] for name,path in paths.items()):
                missing.append(dict(component='original_features',reason=f'{label} fingerprint mismatch; not used',required='Matching 08b input files'))
                continue
            verified=(data,samples,paths,label)
            break
    if feature_path:
        path=Path(feature_path).expanduser().resolve()
        if not path.is_file(): raise FileNotFoundError(f'No feature_table at {path}')
        features=_feature_identity(_csv(path),state)
        provenance['feature_table']={'path':str(path),'sha256':_sha(path),'verification':'explicit supplied table; keys/labels checked, not an original feature-file fingerprint'}
    elif verified:
        data,samples,paths,label=verified
        repo=Path(config['repo_root']).expanduser().resolve()
        modules=('features.py','features_v1.py')
        if all((repo/'ml'/name).is_file() and _sha(repo/'ml'/name)==state.get('code_sha256',{}).get(name) for name in modules):
            sys.path.insert(0,str(repo/'ml'))
            for name in ('features','features_v1'): sys.modules.pop(name,None)
            importlib.invalidate_caches()
            from features import check_estimates
            from features_v1 import build_table_v1,check_source_estimates
            rebuilt=build_table_v1(str(samples),str(data))
            check_estimates(rebuilt);check_source_estimates(rebuilt)
            features=_feature_identity(rebuilt,state)
            if any(_sha(path)!=expected[name] for name,path in paths.items()): raise ValueError('Inputs changed while rebuilding diagnostic features')
            provenance['feature_table']={'source':label,'verification':'original input and feature-code hashes matched; no model inference'}
        else:
            missing.append(dict(component='original_features',reason='Matching original features.py/features_v1.py unavailable',required='repo_root containing original feature code, or feature_table CSV'))
    if features is None:
        missing.append(dict(component='original_features',reason='No available matching snapshot or supplied feature table',required='inputs/data/{queries-Ans.txt,queries-meta.csv}, inputs/run/{rows.csv,src_*.txt,arm_*.txt}, or feature_table CSV'))
    if metadata_path:
        path=Path(metadata_path).expanduser().resolve()
        if not path.is_file(): raise FileNotFoundError(f'No metadata_file at {path}')
        wanted=expected.get('data/queries-meta.csv')
        if wanted and _sha(path)!=wanted: raise ValueError('metadata_file fingerprint differs from original08b queries-meta.csv')
        metadata=_csv(path);provenance['metadata']={'path':str(path),'sha256':_sha(path)}
    elif verified:
        path=verified[0]/'queries-meta.csv'
        metadata=_csv(path);provenance['metadata']={'path':str(path),'sha256':_sha(path)}
    elif source.directory and (source.directory/'inputs/data/queries-meta.csv').is_file():
        path=source.directory/'inputs/data/queries-meta.csv'
        if expected.get('data/queries-meta.csv') and _sha(path)!=expected['data/queries-meta.csv']: raise ValueError('Saved metadata fingerprint differs')
        metadata=_csv(path);provenance['metadata']={'path':str(path),'sha256':_sha(path)}
    if metadata is not None:
        if not {'qId','scale'}.issubset(metadata) or metadata.qId.duplicated().any(): raise ValueError('Metadata needs unique qId and existing scale')
        if metadata[['qId','scale']].isna().any().any(): raise ValueError('Missing metadata query IDs or scales')
        metadata=metadata[[name for name in ('qId','scale','width','height') if name in metadata]].copy()
    elif features is not None and 'scale' in features:
        columns=[name for name in ('qId','scale','width','height') if name in features]
        metadata=features[columns].drop_duplicates()
        if metadata.qId.duplicated().any(): raise ValueError('Metadata scale/context varies by merge rate')
        provenance['metadata']={'source':'existing scale in supplied/original feature table'}
    else:
        missing.append(dict(component='query_size',reason='Original query-size scale/metadata unavailable',required='Original queries-meta.csv (qId,scale,width,height)'))
    if features is not None:
        present=[name for name in FEATURES_18 if name in features]
        absent=[name for name in FEATURES_18 if name not in features]
        if absent: missing.append(dict(component='feature_columns',reason='Unavailable: '+','.join(absent),required='Original18-feature table'))
        if splits is None and features.qId.nunique()==int(state.get('unique_queries',-1)):
            records=[]
            for seed in SEEDS:
                for label,ids in _canonical_splits(features.qId,seed).items():
                    records.extend(dict(seed=seed,qId=qid,split=label) for qid in ids)
            splits=_validate_splits(pd.DataFrame(records),predictions,state)
            provenance['split_source']='original canonical split reconstructed from complete eligible feature universe and checked against test IDs'
        available=features[['qId','mergeRate']+present+[name for name in ('relErr',) if name in features]].copy()
        check=predictions[['qId','mergeRate','relErr']].drop_duplicates().merge(available,on=['qId','mergeRate'],how='left',validate='one_to_one',suffixes=('','_features'),indicator=True)
        if (check['_merge']!='both').any(): missing.append(dict(component='feature_coverage',reason=f'{int((check["_merge"]!="both").sum())} distinct test query/rate rows have no features',required='Complete original feature table'))
        if 'relErr_features' in check:
            matched=check['_merge'].eq('both')
            if not np.allclose(check.loc[matched,'relErr'],check.loc[matched,'relErr_features'],rtol=1e-10,atol=1e-12): raise ValueError('Feature-table true errors disagree with saved predictions')
        features=features[['qId','mergeRate']+present].copy()
    contrast_training=None
    training_path=str(config.get('training_table','')).strip()
    if training_path:
        path=Path(training_path).expanduser().resolve()
        if not path.is_file(): raise FileNotFoundError(f'No training_table at {path}')
        contrast_training=_csv(path)
        provenance['training_table']={'path':str(path),'sha256':_sha(path)}
    elif features is not None: contrast_training=features
    if splits is None:
        missing.append(dict(component='training_split',reason='Original training IDs cannot be verified from ZIP-only test predictions',required='Saved splits.csv or complete eligible feature table'))
    if features is None or 'src_edge_contrast' not in features:
        missing.append(dict(component='source_contrast',reason='Original source contrast values unavailable; test-based bins are forbidden',required='Original src_*.txt plus feature inputs, or feature_table containing src_edge_contrast'))
    if metadata is not None:
        unknown=set(predictions.qId)-set(metadata.qId)
        if unknown: missing.append(dict(component='metadata_coverage',reason=f'{len(unknown)} test query IDs have no metadata',required='Complete original queries-meta.csv'))
    return dict(features=features,metadata=metadata,splits=splits,contrast_training=contrast_training,
                missing=missing,provenance=provenance)

def add_context(paired,diagnostics):
    out=paired.copy()
    if diagnostics['features'] is not None:
        out=out.merge(diagnostics['features'],on=['qId','mergeRate'],how='left',validate='many_to_one')
    if diagnostics['metadata'] is not None:
        out=out.merge(diagnostics['metadata'],on='qId',how='left',validate='many_to_one')
    for name in FEATURES_18+['scale','width','height']:
        if name not in out: out[name]=np.nan
    return out

def add_contrast_bins(paired,diagnostics):
    out=paired.copy();out['source_contrast_bin']=pd.Series(pd.NA,index=out.index,dtype='object')
    records=[]
    training=diagnostics['contrast_training'];splits=diagnostics['splits']
    if training is None or splits is None or 'src_edge_contrast' not in training or 'qId' not in training:
        return out,pd.DataFrame(columns=['seed','training_queries','interior_edges','method'])
    for seed in SEEDS:
        ids=set(splits.loc[splits.seed.eq(seed)&splits.split.eq('train'),'qId'])
        candidates=training[training.qId.isin(ids)].copy()
        if 'seed' in candidates: candidates=candidates[candidates.seed.eq(seed)]
        if 'split' in candidates and not candidates['split'].eq('train').all(): raise ValueError('training_table contains a nontraining label for selected train IDs')
        if set(candidates.qId)!=ids:
            diagnostics['missing'].append(dict(component=f'source_contrast_bins_seed_{seed}',reason='Not all verified training IDs have contrast values',required='Complete source contrast for this seed training cohort'))
            continue
        if not np.isfinite(candidates.src_edge_contrast).all(): raise ValueError('Nonfinite training source contrast')
        if candidates.groupby('qId').src_edge_contrast.nunique().gt(1).any(): raise ValueError('Source contrast varies by merge rate for a query')
        values=candidates.drop_duplicates('qId').src_edge_contrast.to_numpy()
        edges=np.unique(np.quantile(values,[.25,.5,.75]))
        edges=edges[(edges>values.min())&(edges<values.max())]
        # Equal-to-edge values stay in the lower bin. Boundaries never use test data.
        mask=out.seed.eq(seed)&out.src_edge_contrast.notna()
        out.loc[mask,'source_contrast_bin']=['C'+str(index+1) for index in np.searchsorted(edges,out.loc[mask,'src_edge_contrast'],side='left')]
        records.append(dict(seed=seed,training_queries=len(values),interior_edges=json.dumps(edges.tolist()),
                            method='per-seed unique training qId quartiles; exterior bounds -inf/+inf; right-closed bins'))
    return out,pd.DataFrame(records,columns=['seed','training_queries','interior_edges','method'])


## Paired outcomes, groups and severity

Class-added cases are accepted only by classification. Class-lost cases are accepted only by
regression. Both good and bad counts are reported for each direction; the acceptance sets are not
assumed nested. Group risk uses bad accepted / accepted; good-answer recall uses good accepted /
all genuinely good cases in that group. Zero denominators remain undefined.

Severity bins are (25%,30%], (30%,50%], (50%,100%], >100%. The label boundary stays25%.
Representative cases are intentionally selected diagnostics, not a random estimate of prevalence.


In [ ]:
def _metrics(part,model,config):
    accepted=part['accepted_'+model].to_numpy(bool)
    good=part.good.to_numpy(bool)
    n=len(part);a=int(accepted.sum());bad=int((accepted&~good).sum());good_n=int(good.sum());good_a=int((accepted&good).sum())
    support=[(len(group),int(group.qId.nunique()),int(group['accepted_'+model].sum())) for _,group in part.groupby('seed')]
    min_cases=min((row[0] for row in support),default=0)
    min_unique=min((row[1] for row in support),default=0)
    min_accepted=min((row[2] for row in support),default=0)
    return dict(total_cases=n,unique_queries=int(part.qId.nunique()),accepted_count=a,violation_count=bad,
                true_good_count=good_n,good_accepted_count=good_a,acceptance_ratio=_ratio(a,n),
                accepted_violation_rate=_ratio(bad,a),good_acceptance_ratio=_ratio(good_a,good_n),
                min_seed_cases=min_cases,min_seed_unique_queries=min_unique,min_seed_accepted=min_accepted,
                low_support_any_seed=(min_cases<int(config['min_group_cases']) or min_unique<int(config['min_group_cases']) or min_accepted<int(config['min_group_accepted'])),
                low_case_support=n<int(config['min_group_cases']),
                low_unique_query_support=part.qId.nunique()<int(config['min_group_cases']),
                low_accepted_support=a<int(config['min_group_accepted']))

def group_metrics(paired,config):
    specs=[('overall',[]),('merge_rate',['mergeRate'])]
    if paired.scale.notna().any(): specs.extend([('query_size',['scale']),('query_size_x_merge_rate',['scale','mergeRate'])])
    if paired.source_contrast_bin.notna().any(): specs.append(('source_contrast',['source_contrast_bin']))
    records=[]
    for scope,seeds in [('per_seed',[*SEEDS]),('all_seeds_descriptive',[None])]:
        for seed in seeds:
            frame=paired if seed is None else paired[paired.seed.eq(seed)]
            for dimension,columns in specs:
                iterable=[((),frame)] if not columns else frame.groupby(columns,dropna=False,observed=True,sort=True)
                for identity,part in iterable:
                    identity=identity if isinstance(identity,tuple) else (identity,)
                    context={name:value for name,value in zip(columns,identity)}
                    if columns and any(pd.isna(value) for value in identity): continue
                    for model in MODELS:
                        records.append(dict(scope=scope,seed=seed,model_kind=model,dimension=dimension,
                                            **context,**_metrics(part,model,config)))
    return pd.DataFrame(records)

def overlap_table(paired):
    records=[]
    for scope,columns in [('per_seed_rate',['seed','mergeRate']),('per_seed',['seed']),('all_seeds_descriptive',[])]:
        iterable=[((),paired)] if not columns else paired.groupby(columns,sort=True)
        for identity,part in iterable:
            identity=identity if isinstance(identity,tuple) else (identity,)
            row=dict(scope=scope,**dict(zip(columns,identity)),total_cases=len(part),unique_queries=part.qId.nunique())
            for good,label in [(True,'good'),(False,'bad')]:
                for overlap in ('both','regression_only','classification_only','neither'):
                    row[label+'_'+overlap]=int((part.good.eq(good)&part.acceptance_overlap.eq(overlap)).sum())
            row['net_added_good']=row['good_classification_only']-row['good_regression_only']
            row['net_added_bad']=row['bad_classification_only']-row['bad_regression_only']
            records.append(row)
    return pd.DataFrame(records)

def outcome_table(decisions):
    table=decisions.groupby(['model_kind','seed','mergeRate','outcome'],observed=True).size().rename('case_count').reset_index()
    full=pd.MultiIndex.from_product([MODELS,SEEDS,sorted(decisions.mergeRate.unique()),OUTCOMES],names=['model_kind','seed','mergeRate','outcome'])
    return table.set_index(['model_kind','seed','mergeRate','outcome']).reindex(full,fill_value=0).reset_index()

def severity_tables(paired):
    summaries=[];bins=[]
    edges=[.25,.30,.50,1.,np.inf];labels=['(25%,30%]','(30%,50%]','(50%,100%]','>100%']
    for scope,seeds in [('per_seed',[*SEEDS]),('all_seeds_descriptive',[None])]:
        for seed in seeds:
            frame=paired if seed is None else paired[paired.seed.eq(seed)]
            for model in MODELS:
                bad=frame.loc[frame['accepted_'+model]&~frame.good,'relErr']
                summaries.append(dict(scope=scope,seed=seed,model_kind=model,false_accept_count=len(bad),
                                      median_rel_error=bad.median(),p90_rel_error=bad.quantile(.9),max_rel_error=bad.max(),
                                      unique_queries=frame.loc[frame['accepted_'+model]&~frame.good,'qId'].nunique()))
                counts=pd.cut(bad,edges,labels=labels,right=True).value_counts(sort=False)
                if int(counts.sum())!=len(bad): raise ValueError('Severity bins did not partition every violation')
                for label in labels:
                    bins.append(dict(scope=scope,seed=seed,model_kind=model,severity=label,count=int(counts[label]),
                                     share_of_false_accepts=_ratio(counts[label],len(bad))))
    return pd.DataFrame(summaries),pd.DataFrame(bins)

def pick_examples(paired,config):
    specifications=[('both_accept_severe',paired.acceptance_overlap.eq('both')&~paired.good&paired.relErr.gt(1),
                     'Within each seed, highest true relative error >100% accepted by both; seed-balanced round robin preferring unused rates, ties by rate/qId; unique qId.'),
                    ('classification_added_bad',paired.acceptance_overlap.eq('classification_only')&~paired.good,
                     'Within each seed, highest true relative error among classification-only incorrect acceptances; seed-balanced round robin preferring unused rates, ties by rate/qId; unique qId.'),
                    ('both_reject_good',paired.acceptance_overlap.eq('neither')&paired.good,
                     'Within each seed, lowest true relative error among good answers rejected by both; seed-balanced round robin preferring unused rates, ties by rate/qId; unique qId.')]
    rows=[];selection=[];limit=int(config['examples_per_category'])
    for category,mask,rule in specifications:
        eligible=paired[mask].copy()
        ascending=category=='both_reject_good'
        eligible=eligible.sort_values(['relErr','seed','mergeRate','qId'],ascending=[ascending,True,True,True],kind='stable')
        queues={seed:list(part.index) for seed,part in eligible.groupby('seed',sort=True)}
        chosen=[];seen=set();seen_rates=set()
        while queues and len(chosen)<limit:
            for identity in list(queues):
                queue=queues[identity]
                while queue and int(paired.loc[queue[0],'qId']) in seen: queue.pop(0)
                if queue and len(chosen)<limit:
                    position=next((j for j,index in enumerate(queue) if paired.loc[index,'mergeRate'] not in seen_rates and int(paired.loc[index,'qId']) not in seen),0)
                    index=queue.pop(position);seen.add(int(paired.loc[index,'qId']));seen_rates.add(paired.loc[index,'mergeRate']);chosen.append(index)
                if not queue: del queues[identity]
        if chosen:
            picked=paired.loc[chosen].copy();picked.insert(0,'category',category);picked['selection_rule']=rule
            rows.append(picked)
        selection.append(dict(category=category,eligible_cases=len(eligible),eligible_unique_queries=eligible.qId.nunique(),
                              selected_cases=len(chosen),selection_rule=rule))
    columns=['category','seed','qId','mergeRate','scale','width','height','relErr','good','acceptance_overlap',
             'accepted_regression','accepted_classification','score_regression','cutoff_regression',
             'score_classification','cutoff_classification','accept_none_regression','accept_none_classification']+FEATURES_18+['source_contrast_bin','selection_rule']
    examples=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=columns)
    return examples.reindex(columns=columns),pd.DataFrame(selection)


## Three figures, with empirical and supplemental panels kept separate

1. Good/bad acceptance overlap: shared, regression-only and classification-only cases.
2. Severity of incorrectly accepted answers; the displayed counts are repeated evaluation cases.
3. Query-size × merge-rate conditional risk if size metadata exists, otherwise per-rate risk.

Risk heatmaps mark low support with `*`, including low accepted support in any seed even if the
all-seed total is larger; an empty cell has undefined risk. Counts and support flags
remain in CSV so a visually large risk based on a few accepted cases is not interpreted as stable.


In [ ]:
def make_figures(tables,folder,config):
    folder=Path(folder);paths=[]
    colors=['#5b6c87','#3274a1','#e1812c']
    fig,axes=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
    for col,policy in enumerate(POLICIES):
        summary=tables[policy]['overlap'].query("scope == 'all_seeds_descriptive'").iloc[0]
        for row,label in enumerate(['bad','good']):
            ax=axes[row,col];bottom=0
            for overlap,color in zip(['both','regression_only','classification_only'],colors):
                value=int(summary[label+'_'+overlap])
                ax.bar([0],[value],bottom=bottom,label=overlap.replace('_',' '),color=color,width=.55)
                if value: ax.text(0,bottom+value/2,f'{value:,}',ha='center',va='center',fontsize=9,color='white')
                bottom+=value
            ax.text(0,bottom,f'total {bottom:,}',ha='center',va='bottom',fontsize=9)
            ax.set_xticks([0],['Incorrectly accepted' if label=='bad' else 'Correctly accepted'])
            ax.set_title(policy+(' (primary)' if policy=='empirical' else ' (supplement)'))
            ax.set_ylabel('Repeated query × rate × seed cases');ax.margins(y=.15);ax.legend(fontsize=8,loc='upper right')
    fig.suptitle('Which accepted answers are shared or unique to each model? Separate scales for bad/good cases')
    path=folder/'01_acceptance_overlap.png';fig.savefig(path,dpi=170);plt.close(fig);paths.append(path)
    fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
    labels=['(25%,30%]','(30%,50%]','(50%,100%]','>100%']
    for ax,policy in zip(axes,POLICIES):
        data=tables[policy]['severity_bins'].query("scope == 'all_seeds_descriptive'")
        for model,offset,color in zip(MODELS,[-.2,.2],colors[1:]):
            values=data[data.model_kind.eq(model)].set_index('severity').reindex(labels)['count']
            ax.bar(np.arange(4)+offset,values,width=.38,label=model,color=color)
        ax.set_xticks(np.arange(4),labels,rotation=15);ax.set_title(policy);ax.set_ylabel('Incorrectly accepted cases');ax.legend(fontsize=8)
    fig.suptitle('How severe are the incorrect acceptances?')
    path=folder/'02_false_accept_severity.png';fig.savefig(path,dpi=170);plt.close(fig);paths.append(path)
    fig,axes=plt.subplots(2,2,figsize=(13,7),constrained_layout=True)
    for row,policy in enumerate(POLICIES):
        group=tables[policy]['groups'].query("scope == 'all_seeds_descriptive'")
        use_size=group.dimension.eq('query_size_x_merge_rate').any()
        for col,model in enumerate(MODELS):
            ax=axes[row,col]
            data=group[group.model_kind.eq(model)&group.dimension.eq('query_size_x_merge_rate' if use_size else 'merge_rate')].copy()
            if use_size:
                matrix=data.pivot(index='scale',columns='mergeRate',values='accepted_violation_rate').sort_index().sort_index(axis=1)
                low=data.assign(low=data.low_case_support|data.low_unique_query_support|data.low_accepted_support|data.low_support_any_seed).pivot(index='scale',columns='mergeRate',values='low').reindex(index=matrix.index,columns=matrix.columns)
                image=ax.imshow(matrix.to_numpy(float),aspect='auto',vmin=0,vmax=.2,cmap='magma_r')
                ax.set_xticks(range(len(matrix.columns)),[f'{value:.2f}' for value in matrix.columns],rotation=70,fontsize=7)
                ax.set_yticks(range(len(matrix.index)),[str(value) for value in matrix.index]);ax.set_ylabel('Original query-size scale')
                for y in range(len(matrix.index)):
                    for x in range(len(matrix.columns)):
                        if pd.notna(matrix.iloc[y,x]) and bool(low.iloc[y,x]): ax.text(x,y,'*',ha='center',va='center',color='cyan')
                fig.colorbar(image,ax=ax,label='Bad accepted / accepted (saturated at20%)')
            else:
                ax.plot(data.mergeRate,data.accepted_violation_rate,'o-',color=colors[col+1],markersize=3)
                low=data.low_case_support|data.low_unique_query_support|data.low_accepted_support|data.low_support_any_seed
                ax.scatter(data.loc[low,'mergeRate'],data.loc[low,'accepted_violation_rate'],marker='x',color='red',label='low support')
                ax.axhline(.05,color='black',linestyle='--',linewidth=1);ax.set_ylabel('Bad accepted / accepted');ax.legend(fontsize=7)
            ax.set_title(policy+' — '+model);ax.set_xlabel('Merge rate')
    fig.suptitle('Where does accepted-answer risk concentrate?  * = low support; blank = undefined')
    path=folder/'03_conditional_risk.png';fig.savefig(path,dpi=170);plt.close(fig);paths.append(path)
    return paths

def write_chinese_analysis(folder,tables,diagnostics,state):
    lines=['# 08c 錯誤放行診斷','',
           '本分析重用08b已保存的預測與calibration門檻，未重訓、未推論、未調整門檻。真實相對誤差≤25%為合格。empirical為主要結果，grid_empirical分開作補充。','',
           '## 已觀察到的結果','']
    for policy in POLICIES:
        aggregate=tables[policy]['groups'].query("scope == 'all_seeds_descriptive' and dimension == 'overall'").set_index('model_kind')
        overlap=tables[policy]['overlap'].query("scope == 'all_seeds_descriptive'").iloc[0]
        label='主要 empirical' if policy=='empirical' else '補充 grid_empirical'
        lines.append(f'### {label}')
        for model in MODELS:
            row=aggregate.loc[model]
            risk=f'{row.accepted_violation_rate:.2%}' if np.isfinite(row.accepted_violation_rate) else '未定義（零接受）'
            lines.append(f'- {model}：接受{int(row.accepted_count):,}筆次，其中違規{int(row.violation_count):,}筆次；接受比例{row.acceptance_ratio:.2%}，接受答案違規率{risk}，合格答案召回率{row.good_acceptance_ratio:.2%}。')
        lines.append(f'- 兩者共同錯誤放行{int(overlap.bad_both):,}筆次；只有回歸錯放{int(overlap.bad_regression_only):,}筆次；只有分類錯放{int(overlap.bad_classification_only):,}筆次。')
        lines.append(f'- 分類新增接受（回歸拒絕）包含合格{int(overlap.good_classification_only):,}、違規{int(overlap.bad_classification_only):,}筆次；同時失去回歸原本接受的合格{int(overlap.good_regression_only):,}、違規{int(overlap.bad_regression_only):,}筆次。不能把淨增加量當成新增集合。')
        severity=tables[policy]['severity'].query("scope == 'all_seeds_descriptive'").set_index('model_kind')
        for model in MODELS:
            r=severity.loc[model]
            if r.false_accept_count:
                lines.append(f'- {model}錯放答案真實誤差：中位數{r.median_rel_error:.2%}，p90={r.p90_rel_error:.2%}。')
        sizes=tables[policy]['groups'].query("scope == 'all_seeds_descriptive' and dimension == 'query_size'")
        eligible=sizes[~sizes.low_case_support&~sizes.low_unique_query_support&~sizes.low_accepted_support&~sizes.low_support_any_seed&sizes.accepted_violation_rate.notna()]
        if len(eligible):
            worst=eligible.sort_values(['accepted_violation_rate','accepted_count'],ascending=False).iloc[0]
            lines.append(f'- 有足夠顯示樣本的size分組中，最高觀察違規率出現在{worst.model_kind}、既有scale={worst.scale}：{int(worst.violation_count)}/{int(worst.accepted_count)}={worst.accepted_violation_rate:.2%}。這是描述性最大值，不是穩定性或因果證據；請對照per-seed結果。')
        band=tables[policy]['severity_bins'].query("scope == 'all_seeds_descriptive'").pivot(index='severity',columns='model_kind',values='count')
        near_reg=int(band.loc['(25%,30%]','regression']);near_class=int(band.loc['(25%,30%]','classification'])
        severe_reg=int(band.loc[['(50%,100%]','>100%'],'regression'].sum())
        severe_class=int(band.loc[['(50%,100%]','>100%'],'classification'].sum())
        lines.append(f'- 25%至30%的錯放：回歸{near_reg:,}、分類{near_class:,}筆次（分類減回歸{near_class-near_reg:+,}）；>50%的錯放：回歸{severe_reg:,}、分類{severe_class:,}筆次。錯放頻率與誤差嚴重程度是不同問題，須分開描述。')
        lines.append('')
    missing=pd.DataFrame(diagnostics['missing'])
    lines+=['## 缺少或無法確認的資料','']
    if len(missing):
        for row in missing.drop_duplicates().itertuples(index=False): lines.append(f'- {row.component}：{row.reason}。需要：{row.required}。')
    else: lines.append('- 本次要求的分組輸入皆可取得；資料來源與指紋記錄於provenance.json。')
    lines+=['','## 還不能推論什麼','',
            '- 跨seed、merge rate的同一query重複出現，計數是評估筆次，不能當成獨立樣本做顯著性宣稱。per-seed表是主要檢查，all_seeds_descriptive只是重複加權的描述。',
            '- 樣本旗標只提醒低支持度；超過門檻也不代表穩定。零接受時違規率未定義，不能稱為0%達標。',
            '- 代表案例依明確極端案例規則挑選，不是隨機抽樣。不同模型分數尺度不同，不能直接比較數值大小。',
            '- 觀察到特徵群組的錯誤集中，不能證明特徵造成錯誤，也不能直接斷言需要sidecar。',
            '- 這是已看過test的探索性診斷。不能依本次模式修改規則，再用同一份test宣稱改善；真值小於10的查詢與端到端延遲不在本實驗範圍。','',
            '## 最有依據的一個後續實驗','']
    severity=tables['empirical']['severity'].query("scope == 'all_seeds_descriptive'")
    median=float(severity.median_rel_error.median()) if len(severity) else np.nan
    if np.isfinite(median) and median<=.5:
        lines.append('先做一個固定特徵與模型族的選模實驗：只在原training內validation，以低風險接受能力選擇iteration／候選模型，並與原本loss選模對照；保留原calibration方法。理由是本次錯放含有靠近25%邊界的案例，整體loss或AUC未必代表可靠尾端。先在training/validation確認模式，再凍結設定，用新的保留查詢驗證；不根據本次test直接修改門檻。')
    else:
        lines.append('先在原training內validation重做相同的嚴重錯放與群組診斷，確認本次共同或嚴重錯放模式是否重現，再以固定18特徵比較低風險選模與原loss選模；calibration維持不變。選定設定後以新的保留查詢驗證，避免直接針對已看過test加規則或追加來源資訊。')
    Path(folder,'analysis_zh.md').write_text('\n'.join(lines)+'\n')


## Run the diagnostic

Set the source path in widgets and run the cells above, then this cell. The output is a fresh directory
containing audit_counts.csv, input/provenance records, per-policy compact CSVs, three PNGs and a short
Chinese analysis. Detailed decisions/pairs are gzip CSVs when save_decisions=true. A failed audit is
recorded as failed and must be resolved; no partial result is presented as verified.


In [ ]:
def run_analysis(config):
    config={**DEFAULTS,**dict(config)}
    for key in ('min_group_cases','min_group_accepted','examples_per_category'):
        if int(config[key])<1: raise ValueError(f'{key} must be positive')
    output_text=str(config['output_root']).strip()
    if not output_text: raise ValueError('Set a writable personal output_root outside the Git checkout')
    output=Path(output_text).expanduser().resolve()
    repo=Path(config['repo_root']).expanduser().resolve()
    if output==repo or output.is_relative_to(repo): raise ValueError('output_root must be outside the Git checkout')
    output.mkdir(parents=True,exist_ok=True)
    source=SavedSource(config)
    if source.directory and (output==source.directory or output.is_relative_to(source.directory)):
        source.close();raise ValueError('output_root must not be inside the source08b run')
    destination=output/('error-case-analysis-'+uuid4().hex[:12])
    destination.mkdir(exist_ok=False)
    status=dict(status='running',study='08c-error-case-analysis',config=config,source=str(source.path),started=time.time())
    _write_json(destination/'run.json',status)
    try:
        state=source.state('run.json');budget=validate_source_state(state)
        predictions=load_predictions(source,state,budget)
        gates=load_gates(source,state,budget)
        decisions,audit=reconstruct_decisions(predictions,gates)
        audit.to_csv(destination/'audit_counts.csv',index=False)
        paired=pair_decisions(decisions)
        diagnostics=load_diagnostics(config,source,state,predictions)
        paired=add_context(paired,diagnostics)
        paired,bin_edges=add_contrast_bins(paired,diagnostics)
        bin_edges.to_csv(destination/'training_bin_edges.csv',index=False)
        missing=pd.DataFrame(diagnostics['missing'],columns=['component','reason','required']).drop_duplicates()
        missing.to_csv(destination/'missing_inputs.csv',index=False)
        tables={}
        for policy in POLICIES:
            folder=destination/policy;folder.mkdir()
            p=paired[paired.policy.eq(policy)].copy()
            d=decisions[decisions.policy.eq(policy)].copy()
            groups=group_metrics(p,config)
            overlap=overlap_table(p)
            severity,bins=severity_tables(p)
            examples,selection=pick_examples(p,config)
            tables[policy]=dict(groups=groups,overlap=overlap,severity=severity,severity_bins=bins)
            for name,frame in [('groups',groups),('overlap',overlap),('outcomes',outcome_table(d)),('severity',severity),
                               ('severity_bins',bins),('representative_cases',examples),('representative_selection',selection)]:
                frame.to_csv(folder/(name+'.csv'),index=False)
            if _bool(config['save_decisions']):
                d.to_csv(folder/'decisions.csv.gz',index=False,compression='gzip')
                p.to_csv(folder/'paired_cases.csv.gz',index=False,compression='gzip')
        figures=make_figures(tables,destination,config)
        write_chinese_analysis(destination,tables,diagnostics,state)
        provenance=dict(source=str(source.path),source_git_commit=state.get('git_commit'),source_git_dirty=state.get('git_dirty'),
                        source_input_sha256=state.get('input_sha256'),source_code_sha256=state.get('code_sha256'),
                        artifacts_sha256=source.artifact_hashes,diagnostic_inputs=diagnostics['provenance'],
                        features=FEATURES_18,seeds=SEEDS,policies=POLICIES,epsilon=EPSILON,
                        decisions='strict saved score <= saved cutoff; no tolerance; accept_none rejects all',
                        training_bins='per-seed unique verified trainingqIds only; no test-fitted boundaries',
                        aggregation='per-seed primary; allseed rows repeat query cases and are descriptive only',
                        scope='exploratory already-viewed test; no threshold/model changes; no inference/training')
        _write_json(destination/'provenance.json',provenance)
        status.update(status='complete',completed=time.time(),audit_groups=len(audit),decision_cases=len(decisions),
                      figures=[path.name for path in figures],missing_components=len(missing))
        _write_json(destination/'run.json',status)
        print('Completed08c:',destination)
        print('All saved decision counts matched. Missing-input details:',destination/'missing_inputs.csv')
        print('Read:',destination/'analysis_zh.md')
        return destination
    except Exception as error:
        status.update(status='failed',error=f'{type(error).__name__}: {error}')
        _write_json(destination/'run.json',status)
        raise
    finally: source.close()

CONFIG={key:get(key) for key in DEFAULTS}
result_dir=run_analysis(CONFIG)


## Read the results

Start with `analysis_zh.md`, then empirical/groups.csv and empirical/overlap.csv. Inspect per-seed
rows and support counts before describing a pattern. Use grid_empirical/ only as the separately
labeled supplement. Representative cases include both model scores and both original cutoffs;
compare acceptance decisions within each model, not score magnitudes between models.

`training_bin_edges.csv` records every available seed's fixed contrast cutpoints. An empty file means
contrast bins were unavailable; consult missing_inputs.csv. The three PNGs answer overlap, severity
and conditional-risk questions. Saved decisions and paired cases can be inspected without rerunning
training or reading raw source data again. Nothing in this notebook changes the original08b output.


In [ ]:
for policy in POLICIES:
    print('\n',policy,'— primary' if policy=='empirical' else '— supplement')
    frame=_csv(result_dir/policy/'groups.csv')
    summary=frame[frame.scope.eq('all_seeds_descriptive')&frame.dimension.eq('overall')]
    try: display(summary)
    except NameError: print(summary.to_string(index=False))
print('Results:',result_dir)
print((result_dir/'analysis_zh.md').read_text())
